# Solutions

:::{admonition} Reference solutions
:class: note
Worked solutions for the short exercises in [05-pandas-exercises.ipynb](05-pandas-exercises.ipynb). Exercise 10, the real-dataset walkthrough, has no solution provided.
:::

## Exercise 1: A DataFrame and a column mean

Build a DataFrame with columns `temp_celsius = [18.2, 17.5, 19.1, 16.8]` and `discharge_m3s = [48.0, 51.2, 47.5, 53.1]`. Print the column dtypes and the mean temperature, rounded to two decimals.

In [ ]:
import pandas as pd
df = pd.DataFrame({"temp_celsius": [18.2, 17.5, 19.1, 16.8],
                   "discharge_m3s": [48.0, 51.2, 47.5, 53.1]})
print(df.dtypes.to_dict())
print(round(df["temp_celsius"].mean(), 2))

## Exercise 2: Write and read a CSV with dates

Build a DataFrame with a `date` column (`"2024-06-01"`, `"2024-06-02"`, `"2024-06-03"`) and a `temp_celsius` column. Write it to `_files/obs.csv` without the index, then read it back with `parse_dates=["date"]` and print the dtypes.

In [ ]:
import pandas as pd
from pathlib import Path

Path("_files").mkdir(exist_ok=True)
frame = pd.DataFrame({"date": ["2024-06-01", "2024-06-02", "2024-06-03"],
                      "temp_celsius": [18.2, 17.5, 19.1]})
frame.to_csv("_files/obs.csv", index=False)
obs = pd.read_csv("_files/obs.csv", parse_dates=["date"])
print(obs.dtypes.to_dict())
print(obs.head())

## Exercise 3: Label, position, and mask

Given

```python
df = pd.DataFrame({"station": ["BAS", "LUG", "JFJ"], "temp_celsius": [18.0, 21.0, -1.0]},
                  index=["a", "b", "c"])
```

print the temperature at label `"b"`, the whole first row by position, and all rows whose temperature is below 0 °C.

In [ ]:
import pandas as pd
df = pd.DataFrame({"station": ["BAS", "LUG", "JFJ"], "temp_celsius": [18.0, 21.0, -1.0]},
                  index=["a", "b", "c"])
print(df.loc["b", "temp_celsius"])     # by label
print(df.iloc[0])                      # by position
print(df[df["temp_celsius"] < 0.0])    # boolean mask

## Exercise 4: Resample and rolling

Build a 40-day daily Series indexed by date (`np.random.default_rng(0)`, mean 18, std 2). Print the monthly means and the first five values of the 3-day rolling mean, both rounded to two decimals.

In [ ]:
import numpy as np
import pandas as pd
dates = pd.date_range("2024-06-01", periods=40, freq="D")
s = pd.Series(18 + np.random.default_rng(0).normal(0, 2, 40), index=dates)
print(s.resample("MS").mean().round(2).tolist())
print(s.rolling(window=3).mean().round(2).head(5).tolist())

## Exercise 5: Group and aggregate

Given

```python
df = pd.DataFrame({"station": ["BAS", "BAS", "LUG", "LUG"],
                   "temp_celsius": [18.0, 19.0, 21.0, 22.0]})
```

compute the mean temperature per station and print it as a dict.

In [ ]:
import pandas as pd
df = pd.DataFrame({"station": ["BAS", "BAS", "LUG", "LUG"],
                   "temp_celsius": [18.0, 19.0, 21.0, 22.0]})
print(df.groupby("station")["temp_celsius"].mean().to_dict())

## Exercise 6: Handle a gap three ways

Given `s = pd.Series([1.0, np.nan, np.nan, 4.0, 5.0])`, print the forward-filled series, the linearly interpolated series, and the NaN-skipping mean. Note in a comment why the three differ.

In [ ]:
import numpy as np
import pandas as pd
s = pd.Series([1.0, np.nan, np.nan, 4.0, 5.0])
print(s.ffill().tolist())          # carries the last known value forward
print(s.interpolate().tolist())    # straight line between known neighbours
print(round(s.mean(), 2))          # mean of the present values only
# ffill repeats 1.0; interpolate ramps 1->4; mean ignores the gaps entirely

## Exercise 7: Join two tables

Given an observations table and a metadata table that share a `station` key, left-join the elevation onto the observations and print the result.

```python
obs = pd.DataFrame({"station": ["BAS", "LUG"], "temp_celsius": [18.0, 21.0]})
meta = pd.DataFrame({"station": ["BAS", "LUG"], "elevation_m": [316, 273]})
```

In [ ]:
import pandas as pd
obs = pd.DataFrame({"station": ["BAS", "LUG"], "temp_celsius": [18.0, 21.0]})
meta = pd.DataFrame({"station": ["BAS", "LUG"], "elevation_m": [316, 273]})
print(obs.merge(meta, on="station", how="left"))

## Exercise 8: A log-scale axis

Right-skewed data — many small values and a few very large ones — is common in nature (flood discharge, earthquake energy, city populations). A linear axis crushes the small values into one corner of the plot.

```python
rng = np.random.default_rng(0)
discharge_m3s = pd.Series(np.exp(rng.normal(3.0, 1.0, 200)))
```

1. Plot a histogram of `discharge_m3s` with `.plot(kind="hist")`, on default (linear) axes.
2. Plot it again, passing `logy=True`. Which view makes the long tail easier to read?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
discharge_m3s = pd.Series(np.exp(rng.normal(3.0, 1.0, 200)))

fig, ax = plt.subplots(figsize=(5, 3))
discharge_m3s.plot(ax=ax, kind="hist", color="tab:blue")
ax.set_xlabel("discharge (m3 s-1)")
ax.set_title("linear count axis")
plt.show()

fig, ax = plt.subplots(figsize=(5, 3))
discharge_m3s.plot(ax=ax, kind="hist", color="tab:blue", logy=True)
ax.set_xlabel("discharge (m3 s-1)")
ax.set_title("log-scale count axis")
plt.show()

# the linear view crushes every bin above the peak into almost nothing; logy=True keeps
# the rare, large-discharge bins visible instead of flattening them against the x-axis

## Exercise 9: Filtered vs. unfiltered histogram

```python
rng = np.random.default_rng(0)
temp_celsius = pd.Series(rng.normal(18.0, 2.0, 200))
temp_celsius[:5] = -999.0     # a stuck sensor: five obviously invalid readings
```

1. Plot a histogram of the raw `temp_celsius` values with `.plot(kind="hist")`. What does the stuck sensor do to the plot?
2. Build a boolean mask that keeps only values above −50 °C, filter the Series with it, and plot the histogram again. In one comment, say which of the two plots you would trust.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
temp_celsius = pd.Series(rng.normal(18.0, 2.0, 200))
temp_celsius[:5] = -999.0

fig, ax = plt.subplots(figsize=(5, 3))
temp_celsius.plot(ax=ax, kind="hist", color="tab:red")
ax.set_xlabel("temperature (°C)")
ax.set_title("raw (unfiltered)")
plt.show()

mask = temp_celsius > -50.0
filtered = temp_celsius[mask]

fig, ax = plt.subplots(figsize=(5, 3))
filtered.plot(ax=ax, kind="hist", color="tab:red")
ax.set_xlabel("temperature (°C)")
ax.set_title("filtered")
plt.show()

# the five stuck readings pull the x-axis out to -999, squashing the real distribution
# into one thin bin; the filtered plot is the one worth trusting